# Phase 7 — C2: the unavoidable-error ceiling

**Why (paper §3.10):** our labels are **daily averages**, but each photo is an **instant**.
Pollution genuinely swings within a day, so even a *perfect* model reading the exact
instantaneous pollution from a photo would be "wrong" versus a daily-average label. That
gap is noise no model can beat — a hard ceiling on achievable R².

We estimate **R²_max = 1 − Var(ε) / Var(y)** and report it explicitly as an **UPPER bound**
(it accounts for *label noise only* — a real model also loses accuracy to limited visual signal,
imbalance and domain shift, so the honest R² can sit well below R²_max with no bug). Two
best-practice refinements: `Var(y)` is the **station-grouped test split's** label variance (the
ceiling is split-specific), and `Var(ε)` is **level-reweighted** — we measure the within-day noise
*curve* v(m) by pollution level from **hourly** reference data (OpenAQ) and reweight it to
PM25Vision's own label mix p(m). We add a station cluster-**bootstrap 95% CI** and a
**2012-vs-2024 breakpoint** sensitivity check. This tells us how much room really remains above the
honest R²=0.22 (and whether the published R²=0.55 is even physically attainable).

## Bootstrap — run this first

This one cell makes the notebook self-contained: it grabs the code from GitHub (if it
isn't already here), installs the libraries, connects Google Drive, and makes our `src`
modules importable. **Set `REPO_URL` to your repository's URL.** It's safe to re-run and
also works on a laptop.

In [ ]:
# === Bootstrap — RUN ME FIRST (set REPO_URL to your repo) ===
REPO_URL = "https://github.com/YOUR_USERNAME/pm25-visual-aq.git"   # <-- EDIT THIS

import os, sys, subprocess

def _find_repo_root():
    # Are we already inside the repo (or just above the notebooks/ folder)?
    for cand in (".", "..", "pm25-visual-aq"):
        if os.path.isdir(os.path.join(cand, "src")):
            return os.path.abspath(cand)
    return None

_root = _find_repo_root()
if _root is None:                       # fresh session: clone the code
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "pm25-visual-aq"], check=True)
    _root = os.path.abspath("pm25-visual-aq")
else:                                   # reused clone (e.g. a stale Kaggle dir): pull the latest
    subprocess.run(["git", "-C", _root, "pull", "--ff-only"], check=False)
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

print("repo root:", _root, "| Colab:", IN_COLAB)

## Get a free OpenAQ API key (2 minutes)

1. Go to **https://explore.openaq.org** → sign up (free).
2. Open your **account → API keys** and copy your key.
3. Paste it below. (It's kept only in this session — don't commit it.)

In [ ]:
OPENAQ_API_KEY = "PASTE_YOUR_OPENAQ_KEY_HERE"   # <-- from explore.openaq.org
assert OPENAQ_API_KEY != "PASTE_YOUR_OPENAQ_KEY_HERE", "Add your OpenAQ API key first." 

## Fetch hourly PM2.5 from a region-stratified sample of stations

We pull ~40 stations' hourly readings over a recent 45-day window, **capping stations per country**
so the sample spreads across regions (not dominated by whichever country OpenAQ lists first). Exact
matching to PM25Vision's stations isn't needed — we want a defensible estimate of the within-day AQI
variance *curve*.

In [ ]:
import datetime as dt
from src import ceiling as CE
to = dt.date.today()
frm = to - dt.timedelta(days=45)
hourly = CE.fetch_openaq_hourly(OPENAQ_API_KEY,
                                date_from=frm.isoformat(), date_to=to.isoformat(),
                                n_locations=40, max_per_country=6)
print("hourly rows fetched:", len(hourly), "| stations:", hourly["location_id"].nunique(),
      "| countries:", hourly["country"].nunique())
hourly.head()

## Compute the ceiling (level-reweighted, split-specific, with a bootstrap CI)

In [ ]:
from src.config import load_config
from src import data, splits as S
import numpy as np
cfg = load_config()

# The ceiling is SPLIT-SPECIFIC: Var(y) = the label variance of the split we headline
# (station-grouped, leakage-safe). p(m) for level-reweighting = the whole dataset's label mix.
_, df = data.load_clean(cfg["data"]["drive_path"], from_disk=True, seed=cfg["seed"])
sp = S.make_splits(df, strategy="station_grouped", seed=cfg["seed"],
                   station_col=cfg["data"]["station_col"], time_col=cfg["data"]["time_col"],
                   lon_col=cfg["data"]["lon_col"], lat_col=cfg["data"]["lat_col"])
var_labels = float(np.var(sp.loc[sp["split"] == "test", "pm25"].to_numpy()))
dataset_labels = df["pm25"].to_numpy()

res = CE.compute_ceiling(hourly, dataset_labels, var_labels, min_hours=18, n_boot=1000)
print("Var(eps) level-reweighted:        %.1f  (std ~ %.1f AQI)" % (res["var_epsilon"], res["var_epsilon"]**0.5))
print("Var(y) station-grouped test:      %.1f  (SD  ~ %.1f AQI)" % (var_labels, var_labels**0.5))
print("--------")
print("R2_max (UPPER bound, label noise only): %.3f  [95%% CI %.3f - %.3f]"
      % (res["R2_max"], res["R2_max_lo"], res["R2_max_hi"]))
print("honest R2 we measured (station-grouped): 0.220   |   published baseline: 0.550")
print("honest interval-width floor: ~%.1f AQI (a 90%% range narrower than this over-claims)" % res["interval_width_floor"])
print("based on %d station-days from %d stations" % (res["n_station_days"], res["n_stations"]))

### The within-day noise curve v(m) + a 2012-vs-2024 breakpoint sensitivity check

In [ ]:
import pandas as pd
from src.aqi import PM25_BREAKPOINTS_2024
print("v(m): within-day AQI variance by pollution level (this is the error-by-band story)")
display(pd.DataFrame(res["curve"])[["band", "n_days", "v", "rmse_within_day", "p_reweighted"]])

res_2024 = CE.compute_ceiling(hourly, dataset_labels, var_labels, min_hours=18,
                              table=PM25_BREAKPOINTS_2024, n_boot=200)
print("R2_max sensitivity to the AQI breakpoint convention:")
print("  historical (the labels' own convention): %.3f" % res["R2_max"])
print("  2024 EPA revision:                        %.3f" % res_2024["R2_max"])

## Save for the paper

In [ ]:
import os, json
os.makedirs(cfg["data"]["outputs_dir"], exist_ok=True)
res["R2_max_2024"] = res_2024["R2_max"]
json.dump(res, open(os.path.join(cfg["data"]["outputs_dir"], "error_ceiling.json"), "w"),
          indent=2, default=float)
print("saved error_ceiling.json  (09_leakage draws its ceiling line from this file)")

## Reading the result

- **R²_max is an UPPER bound** (label noise only). The honest gap between our 0.22 and R²_max is the
  room that *better vision* could recover; the gap between R²_max and 1.0 is forever lost to
  daily-average labels.
- **If R²(random) from `09_leakage` (0.759) exceeds R²_max, that split is *provably* leaky** — no
  honest model can beat the label-noise ceiling, so a score above it can only come from leakage. The
  `error_ceiling.json` saved here is what draws the ceiling line on the leakage-gradient figure.
- The **width floor** is a lower bound on honest 90% interval width: no interval should be narrower
  than the pollution's own within-day spread.
- The **95% CI** and the **2012-vs-2024** sensitivity show the estimate is robust, not a single fragile number.

Paste these numbers back and I'll write them into `docs/RESULTS.md`.

**Next:** `08_abstention.ipynb` (C3 — refusing to answer on unusable inputs).